<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи:
Создать базовый класс Notification в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Notification:
• Атрибуты: ID уведомления (NotificationId), Текст уведомления (MessageText),
Тип уведомления (Type).
• Методы:
o DisplayNotification(): метод для отображения уведомления
пользователю.
o SendNotification(): метод для отправки уведомления.
o GetNotificationDetails(): метод для получения деталей уведомления.

Требования к производным классам:
1. EmailУведомление (EmailNotification): Должно содержать дополнительные
атрибуты, такие как Адрес электронной почты (EmailAddress).
Метод SendNotification() должен быть переопределен для отправки
уведомления по электронной почте.
2. SMSУведомление (SMSNotification): Должно содержать дополнительные
атрибуты, такие как Номер телефона (PhoneNumber).
Метод SendNotification() должен быть переопределен для отправки
уведомления через SMS.
3. PushУведомление (PushNotification) (если требуется третий класс): Должно
содержать дополнительные атрибуты, такие как Платформа (Platform,
например, iOS или Android). Метод DisplayNotification() должен быть
переопределен для отображения уведомления на мобильной платформе.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Делегаты
public delegate void NotificationHandler(Notification notification);
public delegate void NotificationErrorHandler(Notification notification, Exception error);

// Обобщённый делегат фильтрации
public delegate bool NotificationFilter<T>(T notification) where T : Notification;

public class NotificationEventArgs : EventArgs
{
    public Notification Notification { get; }
    public DateTime Timestamp { get; }

    public NotificationEventArgs(Notification notification)
    {
        Notification = notification;
        Timestamp = DateTime.Now;
    }
}

public enum ImportanceLevel { Low, Medium, High, Critical }

public interface INotificationSender
{
    void Send(Notification n);
}

public class ConsoleNotificationSender : INotificationSender
{
    public void Send(Notification n)
    {
        Console.WriteLine($"[DISPATCH] Отправка {n.Type} #{n.NotificationId} -> Получатель: {n.Recipient}");
        n.IsRead = true;
        n.Status = "Отправлено (через диспетчер)";
    }
}

public abstract class Notification
{
    public int NotificationId { get; set; }
    public string MessageText { get; set; }
    public string Type { get; set; }
    public DateTime CreatedAt { get; set; }
    public bool IsRead { get; set; }
    public string Priority { get; set; }
    public string Recipient { get; set; }
    public string Sender { get; set; }
    public string Status { get; set; }
    public List<string> Tags { get; set; } = new List<string>();
    public DateTime? Expiration { get; set; }
    public int RetryCount { get; set; } = 0;
    public Dictionary<string, string> Metadata { get; set; } = new Dictionary<string, string>();
    public ImportanceLevel Importance { get; set; } = ImportanceLevel.Medium;
    public Guid CorrelationId { get; set; } = Guid.NewGuid();
    public string Channel { get; set; } = "default";
    public bool IsMuted { get; set; } = false;
    public string Language { get; set; } = "ru-RU";

    // События 
    public event EventHandler<NotificationEventArgs> OnCreated;
    public event EventHandler<NotificationEventArgs> OnSent;
    public event EventHandler<NotificationEventArgs> OnRead;
    public event EventHandler<NotificationEventArgs> OnStatusChanged;
    public event EventHandler<NotificationEventArgs> OnArchived;

    protected virtual void RaiseArchived()
    {
        OnArchived?.Invoke(this, new NotificationEventArgs(this));
    }

    // Коллекции
    protected List<string> StatusHistory { get; } = new List<string>();
    protected Queue<DateTime> SendAttempts { get; } = new Queue<DateTime>();
    protected Dictionary<string, object> ExtendedProperties { get; } = new Dictionary<string, object>();

    public IReadOnlyList<string> GetStatusHistory() => StatusHistory.AsReadOnly();
    public IEnumerable<DateTime> GetSendAttempts() => SendAttempts.ToList();
    public IDictionary<string, object> GetExtendedProperties() => new Dictionary<string, object>(ExtendedProperties);

    protected virtual void OnStatusHistoryUpdated(string status)
    {
        StatusHistory.Add($"{DateTime.Now:O}: {status}");
        OnStatusChanged?.Invoke(this, new NotificationEventArgs(this));
    }

    protected INotificationSender NotificationSender { get; private set; }

    public Notification()
    {
        CreatedAt = DateTime.Now;
        OnCreated?.Invoke(this, new NotificationEventArgs(this));
    }

    public Notification(INotificationSender sender) : this()
    {
        NotificationSender = sender;
    }

    public Notification(int notificationId, string messageText, string type, string priority, string recipient, string sender, string status)
        : this()
    {
        NotificationId = notificationId;
        MessageText = messageText;
        Type = type;
        IsRead = false;
        Priority = priority;
        Recipient = recipient;
        Sender = sender;
        Status = status;
    }

    public void SetNotificationSender(INotificationSender sender)
    {
        NotificationSender = sender;
    }

    public virtual void DisplayNotification()
    {
        Console.WriteLine($"[Уведомление] {MessageText} (Приоритет: {Priority}, Получатель: {Recipient}, Статус: {Status})");
    }

    public virtual void DisplayNotification(bool showMetadata)
    {
        DisplayNotification();
        if (showMetadata && Metadata.Any())
        {
            Console.WriteLine("Метаданные:");
            foreach (var kv in Metadata) Console.WriteLine($" - {kv.Key}: {kv.Value}");
        }
    }

    public virtual void DisplayNotification(string format)
    {
        if (format == "короткий")
            Console.WriteLine($"[{Type}] {MessageText}");
        else
            DisplayNotification();
    }

    public virtual void SendNotification()
    {
        try
        {
            if (NotificationSender != null)
            {
                NotificationSender.Send(this);
                SendAttempts.Enqueue(DateTime.Now);
                OnStatusHistoryUpdated("Отправлено (через диспетчер)");
                OnSent?.Invoke(this, new NotificationEventArgs(this));
                return;
            }

            Status = "Отправлено";
            SendAttempts.Enqueue(DateTime.Now);
            OnStatusHistoryUpdated("Отправлено");
            OnSent?.Invoke(this, new NotificationEventArgs(this));
        }
        catch (Exception ex)
        {
            Status = "Ошибка";
            OnStatusHistoryUpdated($"Ошибка: {ex.Message}");
            throw;
        }
    }

    public virtual void SendNotification(bool async, int maxRetries = 1)
    {
        if (async) Console.WriteLine("Отправка уведомления в фоне...");
        int attempts = 0;
        while (attempts < maxRetries)
        {
            attempts++;
            SendNotification();
            if (Status.Contains("Отправлен")) break;
            RetryCount++;
        }
    }

    public virtual string GetNotificationDetails()
    {
        return $"ID: {NotificationId}, Тип: {Type}, Текст: {MessageText}, Время: {CreatedAt}, Прочитано: {IsRead}, Приоритет: {Priority}, Получатель: {Recipient}, Отправитель: {Sender}, Статус: {Status}";
    }

    public virtual string GetNotificationDetails(bool includeMetadata)
    {
        var baseDetails = GetNotificationDetails();
        if (!includeMetadata || !Metadata.Any()) return baseDetails;
        var meta = string.Join(", ", Metadata.Select(kv => $"{kv.Key}={kv.Value}"));
        return baseDetails + $", Метаданные: {meta}";
    }

    public void MarkAsRead()
    {
        IsRead = true;
        Status = "Прочитано";
        OnStatusHistoryUpdated("Прочитано");
        OnRead?.Invoke(this, new NotificationEventArgs(this));
    }

    public void ChangePriority(string newPriority) => Priority = newPriority;
    public void ChangeRecipient(string newRecipient) => Recipient = newRecipient;
    public void ChangeStatus(string newStatus)
    {
        Status = newStatus;
        OnStatusHistoryUpdated(newStatus);
    }

    public void ScheduleSend(DateTime when)
    {
        Console.WriteLine($"Уведомление {NotificationId} запланировано на {when}.");
        Metadata["ScheduledAt"] = when.ToString("o");
    }

    public virtual bool RetrySend()
    {
        RetryCount++;
        Console.WriteLine($"Повторная попытка отправки ({RetryCount}) для {NotificationId}");
        SendNotification();
        return Status.Contains("Отправлен");
    }

    public string SerializeMetadata() => string.Join(";", Metadata.Select(kv => $"{kv.Key}={kv.Value}"));

    public void Cancel()
    {
        Status = "Отменено";
        OnStatusHistoryUpdated("Отменено");
        Console.WriteLine($"Уведомление {NotificationId} отменено.");
    }

    public void Snooze(TimeSpan duration)
    {
        var when = DateTime.Now.Add(duration);
        ScheduleSend(when);
        Console.WriteLine($"Уведомление {NotificationId} отложено на {duration}.");
    }

    public bool Validate() => NotificationId > 0 && !string.IsNullOrEmpty(MessageText);

    public Notification CloneShallow() => (Notification)MemberwiseClone();
}

public interface ILoggable { void Log(); }
public interface IArchivable { void Archive(); }

// Новый менеджер уведомлений
public class NotificationManager<T> where T : Notification
{
    // использование коллекций 
    private readonly List<T> _items = new List<T>();
    private readonly Dictionary<string, List<T>> _groupedItems = new Dictionary<string, List<T>>();
    private readonly INotificationSender _sender;

    public event EventHandler<NotificationEventArgs> OnNotificationAdded;
    public event EventHandler<NotificationEventArgs> OnNotificationRemoved;

    public NotificationManager(INotificationSender sender = null)
    {
        _sender = sender;
    }

    public void Add(T item)
    {
        if (item == null) return;
        _items.Add(item);

        if (!_groupedItems.ContainsKey(item.Type))
            _groupedItems[item.Type] = new List<T>();

        _groupedItems[item.Type].Add(item);
        OnNotificationAdded?.Invoke(this, new NotificationEventArgs(item));
    }

    public void Remove(T item)
    {
        if (item == null) return;
        if (_items.Remove(item))
        {
            if (_groupedItems.ContainsKey(item.Type))
                _groupedItems[item.Type].Remove(item);
            OnNotificationRemoved?.Invoke(this, new NotificationEventArgs(item));
        }
    }

    public bool RemoveById(int id)
    {
        var it = _items.FirstOrDefault(x => x.NotificationId == id);
        if (it == null) return false;
        Remove(it);
        return true;
    }

    public T GetById(int id) => _items.FirstOrDefault(x => x.NotificationId == id);
    public IEnumerable<T> FilterByTag(string tag) => _items.Where(n => n.Tags.Contains(tag));
    public IEnumerable<T> FilterByRecipient(string recipient) => _items.Where(n => n.Recipient == recipient);

    public IEnumerable<T> Filter(NotificationFilter<T> filter) => _items.Where(x => filter(x));
    public IEnumerable<T> GetByType(string type) => _groupedItems.ContainsKey(type) ? _groupedItems[type].AsReadOnly() : Enumerable.Empty<T>();

    public void SendAll(Func<T, bool> predicate = null)
    {
        var list = predicate == null ? _items : _items.Where(predicate);
        foreach (var n in list)
        {
            if (_sender != null) n.SetNotificationSender(_sender);
            n.SendNotification();
        }
    }

    public void SendFiltered(NotificationFilter<T> filter)
    {
        foreach (var item in Filter(filter))
        {
            if (_sender != null) item.SetNotificationSender(_sender);
            item.SendNotification();
        }
    }

    public void ArchiveAll()
    {
        foreach (var n in _items.OfType<IArchivable>())
            ((IArchivable)n).Archive();
    }
}

// EmailNotification
public class EmailNotification : Notification, ILoggable, IArchivable
{
    public string EmailAddress { get; set; }
    public string Subject { get; set; }
    public string CC { get; set; }
    public int AttachmentsCount { get; set; }
    public bool IsHtml { get; set; } = false;
    public string ReplyTo { get; set; }
    public int ImportanceScore { get; set; } = 5;

    public bool IsTracked { get; set; } = false;
    public int BounceCount { get; set; } = 0;
    public string UnsubscribeLink { get; set; }

    public EmailNotification() { }

    public EmailNotification(int notificationId, string messageText, string emailAddress, string subject, string cc, int attachmentsCount, string priority, string recipient, string sender, string status)
        : base(notificationId, messageText, "Email", priority, recipient, sender, status)
    {
        EmailAddress = emailAddress;
        Subject = subject;
        CC = cc;
        AttachmentsCount = attachmentsCount;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"Email отправлен на {EmailAddress}: {MessageText} (Тема: {Subject}, CC: {CC}, Вложений: {AttachmentsCount}, HTML: {IsHtml})");
        IsRead = true;
        Status = "Email отправлен";
        base.SendNotification();
    }

    public void SendNotification(string customSubject)
    {
        var old = Subject;
        Subject = customSubject;
        SendNotification();
        Subject = old;
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Email: {EmailAddress}, Тема: {Subject}, CC: {CC}, Вложений: {AttachmentsCount}, HTML: {IsHtml}";
    }

    public void ForwardTo(EmailNotification other)
    {
        other.MessageText = this.MessageText;
        Console.WriteLine($"Уведомление переслано с {EmailAddress} на {other.EmailAddress}");
    }

    void ILoggable.Log()
    {
        Console.WriteLine($"[LOG] Email: {EmailAddress}, Subject: {Subject}, CC: {CC}, Attachments: {AttachmentsCount}, Tracked: {IsTracked}");
    }

    void IArchivable.Archive()
    {
        Status = "Архивировано";
        OnStatusHistoryUpdated("Архивировано");
        RaiseArchived();
        Console.WriteLine($"Email {NotificationId} архивирован.");
    }

    public void AddTag(string tag)
    {
        if (!Tags.Contains(tag)) Tags.Add(tag);
    }

    public void TrackOpen()
    {
        if (IsTracked) Console.WriteLine($"Email {NotificationId} открыт (трек установлен).");
        else Console.WriteLine($"Email {NotificationId} не трекается.");
    }

    public bool ValidateAddress() => !string.IsNullOrWhiteSpace(EmailAddress) && EmailAddress.Contains("@");
}

// SMSNotification
public class SMSNotification : Notification, ILoggable, IArchivable
{
    public string PhoneNumber { get; set; }
    public string Operator { get; set; }
    public int SmsLength { get; set; }
    public bool IsInternational { get; set; }
    public string CarrierCode { get; set; }
    public bool IsUnicode { get; set; }
    public string RegionCode { get; set; }

    public bool IsShortCode { get; set; } = false;
    public decimal Cost { get; set; } = 0m;
    public string DeliveryReceiptId { get; set; }

    public SMSNotification() { }

    public SMSNotification(int notificationId, string messageText, string phoneNumber, string operatorName, int smsLength, bool isInternational, string priority, string recipient, string sender, string status)
        : base(notificationId, messageText, "SMS", priority, recipient, sender, status)
    {
        PhoneNumber = phoneNumber;
        Operator = operatorName;
        SmsLength = smsLength;
        IsInternational = isInternational;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"SMS отправлено на {PhoneNumber}: {MessageText} (Оператор: {Operator}, Длина: {SmsLength}, Международный: {IsInternational}, Unicode: {IsUnicode})");
        IsRead = true;
        Status = "SMS отправлено";
        base.SendNotification();
    }

    public void SendNotification(string countryCode)
    {
        PhoneNumber = countryCode + PhoneNumber;
        SendNotification();
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Телефон: {PhoneNumber}, Оператор: {Operator}, Длина: {SmsLength}, Международный: {IsInternational}, Unicode: {IsUnicode}";
    }

    public void CopyTo(SMSNotification other)
    {
        other.MessageText = this.MessageText;
        Console.WriteLine($"SMS уведомление скопировано с {PhoneNumber} на {other.PhoneNumber}");
    }

    void ILoggable.Log()
    {
        Console.WriteLine($"[LOG] SMS: {PhoneNumber}, Operator: {Operator}, Length: {SmsLength}, International: {IsInternational}, Cost: {Cost}");
    }

    void IArchivable.Archive()
    {
        Status = "Архивировано";
        OnStatusHistoryUpdated("Архивировано");
        RaiseArchived();
        Console.WriteLine($"SMS {NotificationId} архивировано.");
    }

    public void RequestDeliveryReceipt()
    {
        DeliveryReceiptId = Guid.NewGuid().ToString();
        Console.WriteLine($"Запрошен delivery receipt: {DeliveryReceiptId} для SMS {NotificationId}");
    }

    public void FormatForOperator()
    {
        if (!string.IsNullOrEmpty(RegionCode)) PhoneNumber = RegionCode + PhoneNumber;
        Console.WriteLine($"Phone formatted for operator {Operator}: {PhoneNumber}");
    }
}

// PushNotification
public class PushNotification : Notification, ILoggable, IArchivable
{
    public string Platform { get; set; }
    public string AppName { get; set; }
    public string DeviceId { get; set; }
    public bool IsSilent { get; set; }
    public int BadgeCount { get; set; }
    public string Sound { get; set; }
    public string Category { get; set; }

    public TimeSpan TimeToLive { get; set; } = TimeSpan.FromHours(1);
    public string DeepLink { get; set; }
    public bool IsCritical { get; set; } = false;

    public PushNotification() { }

    public PushNotification(int notificationId, string messageText, string platform, string appName, string deviceId, bool isSilent, string priority, string recipient, string sender, string status)
        : base(notificationId, messageText, "Push", priority, recipient, sender, status)
    {
        Platform = platform;
        AppName = appName;
        DeviceId = deviceId;
        IsSilent = isSilent;
    }

    public override void DisplayNotification()
    {
        Console.WriteLine($"Push на {Platform} ({AppName}, Device: {DeviceId}, Silent: {IsSilent}, Badge: {BadgeCount}): {MessageText}");
    }

    public void DisplayNotification(string userLocale, bool showBadge)
    {
        var prefix = userLocale == "ru-RU" ? "[RU]" : "[EN]";
        Console.WriteLine($"{prefix} Push на {Platform} ({AppName}): {MessageText}");
        if (showBadge) Console.WriteLine($"Badge: {BadgeCount}");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Платформа: {Platform}, Приложение: {AppName}, DeviceId: {DeviceId}, Silent: {IsSilent}, Badge: {BadgeCount}";
    }

    public void SyncWith(PushNotification other)
    {
        other.MessageText = this.MessageText;
        Console.WriteLine($"Push уведомление синхронизировано между {Platform} и {other.Platform}");
    }

    void ILoggable.Log()
    {
        Console.WriteLine($"[LOG] Push: {Platform}, App: {AppName}, DeviceId: {DeviceId}, Silent: {IsSilent}, TTL: {TimeToLive}");
    }

    void IArchivable.Archive()
    {
        Status = "Архивировано";
        OnStatusHistoryUpdated("Архивировано");
        RaiseArchived();
        Console.WriteLine($"Push {NotificationId} архивировано.");
    }

    public void WakeDevice() => Console.WriteLine($"Посылаем wake-up для {DeviceId} (Platform {Platform})");
    public void Localize(string locale)
    {
        Language = locale;
        Console.WriteLine($"Push {NotificationId} локализован для {locale}");
    }
}

// EmailMarketingNotification
public class EmailMarketingNotification : EmailNotification
{
    public string CampaignName { get; set; }
    public int TargetAudienceSize { get; set; }
    public bool IsAutomated { get; set; }
    public string MarketingChannel { get; set; }

    public DateTime CampaignStart { get; set; }
    public DateTime CampaignEnd { get; set; }
    public double ConversionRate { get; set; } = 0.0;

    public EmailMarketingNotification(
        int notificationId,
        string messageText,
        string emailAddress,
        string subject,
        string cc,
        int attachmentsCount,
        string priority,
        string recipient,
        string sender,
        string status,
        string campaignName,
        int targetAudienceSize,
        bool isAutomated,
        string marketingChannel)
        : base(notificationId, messageText, emailAddress, subject, cc, attachmentsCount, priority, recipient, sender, status)
    {
        CampaignName = campaignName;
        TargetAudienceSize = targetAudienceSize;
        IsAutomated = isAutomated;
        MarketingChannel = marketingChannel;
    }

    public override void SendNotification()
    {
        base.SendNotification();
        Console.WriteLine($"Маркетинговая кампания: {CampaignName}, Канал: {MarketingChannel}, Автоматизация: {IsAutomated}, Аудитория: {TargetAudienceSize}");
    }

    public void ScheduleCampaign(DateTime date)
    {
        CampaignStart = date;
        Console.WriteLine($"Кампания '{CampaignName}' запланирована на {date}");
    }

    public void AnalyzeResults()
    {
        ConversionRate = new Random().NextDouble() * 0.2;
        Console.WriteLine($"Анализ результатов кампании '{CampaignName}' завершен. ConversionRate={ConversionRate:P2}");
    }

    public List<string> ExportAudience()
    {
        Console.WriteLine($"Экспорт аудитории ({TargetAudienceSize}) для кампании '{CampaignName}'");
        return Enumerable.Range(1, TargetAudienceSize).Select(i => $"user{i}@example.com").ToList();
    }
}

// Пример использования
public static class Demo
{
    public static void Run()
    {
        var email1 = new EmailNotification(1, "Проверьте вашу почту!", "user@example.com", "Входящее", "admin@site.com", 2, "Высокий", "user@example.com", "admin@site.com", "Создано");
        email1.Tags.Add("important");
        email1.Metadata["Folder"] = "Inbox";
        email1.IsHtml = true;
        email1.IsTracked = true;
        email1.UnsubscribeLink = "https://site.com/unsub";
        email1.SendNotification("СПЕЦИАЛЬНАЯ ТЕМА");
        ((ILoggable)email1).Log();
        ((IArchivable)email1).Archive();

        var sms1 = new SMSNotification(3, "Ваш код: 1234", "+79991234567", "МТС", 12, false, "Средний", "+79991234567", "system", "Создано");
        sms1.Tags.Add("auth");
        sms1.IsUnicode = false;
        sms1.Cost = 0.05m;
        sms1.SendNotification();
        sms1.SendNotification("+7");
        sms1.RequestDeliveryReceipt();
        ((ILoggable)sms1).Log();
        ((IArchivable)sms1).Archive();

        var push1 = new PushNotification(5, "Новое обновление доступно!", "Android", "MyApp", "dev123", false, "Высокий", "dev123", "system", "Создано");
        push1.BadgeCount = 3;
        push1.DisplayNotification();
        push1.DisplayNotification("ru-RU", true);
        push1.WakeDevice();
        push1.Localize("ru-RU");
        push1.SendNotification();
        ((ILoggable)push1).Log();
        ((IArchivable)push1).Archive();

        var marketingEmail = new EmailMarketingNotification(
            10, "Скидки только сегодня!", "client@site.com", "Акция!", "marketing@site.com", 1, "Высокий", "client@site.com", "marketing@site.com", "Создано",
            "Весенний Sale", 50, true, "Email");
        marketingEmail.ScheduleCampaign(DateTime.Now.AddDays(1));
        marketingEmail.SendNotification();
        marketingEmail.AnalyzeResults();
        ((ILoggable)marketingEmail).Log();
        ((IArchivable)marketingEmail).Archive();

        var dispatcher = new ConsoleNotificationSender();
        var manager = new NotificationManager<Notification>(dispatcher);
        manager.OnNotificationAdded += (s, e) => Console.WriteLine($"Добавлено уведомление: {e.Notification.NotificationId}");
        manager.OnNotificationRemoved += (s, e) => Console.WriteLine($"Удалено уведомление: {e.Notification.NotificationId}");

        var email = new EmailNotification(11, "Test", "test@test.com", "Subject", "", 0, "High", "recipient", "sender", "New");
        email.OnSent += (s, e) => Console.WriteLine($"Email {e.Notification.NotificationId} отправлен");
        var sms = new SMSNotification(12, "Test SMS", "+1234567890", "Operator", 10, false, "Medium", "recipient", "sender", "New");
        sms.OnSent += (s, e) => Console.WriteLine($"SMS {e.Notification.NotificationId} отправлен");

        manager.Add(email);
        manager.Add(sms);

        manager.SendFiltered(n => n.Priority == "High");
        foreach (var status in email.GetStatusHistory()) Console.WriteLine(status);
    }
}

Demo.Run();

Email отправлен на user@example.com: Проверьте вашу почту! (Тема: СПЕЦИАЛЬНАЯ ТЕМА, CC: admin@site.com, Вложений: 2, HTML: True)
[LOG] Email: user@example.com, Subject: Входящее, CC: admin@site.com, Attachments: 2, Tracked: True
Email 1 архивирован.
SMS отправлено на +79991234567: Ваш код: 1234 (Оператор: МТС, Длина: 12, Международный: False, Unicode: False)
SMS отправлено на +7+79991234567: Ваш код: 1234 (Оператор: МТС, Длина: 12, Международный: False, Unicode: False)
Запрошен delivery receipt: d35ac8b1-6dd9-4437-baf1-094b22d3e911 для SMS 3
[LOG] SMS: +7+79991234567, Operator: МТС, Length: 12, International: False, Cost: 0.05
SMS 3 архивировано.
Push на Android (MyApp, Device: dev123, Silent: False, Badge: 3): Новое обновление доступно!
[RU] Push на Android (MyApp): Новое обновление доступно!
Badge: 3
Посылаем wake-up для dev123 (Platform Android)
Push 5 локализован для ru-RU
[LOG] Push: Android, App: MyApp, DeviceId: dev123, Silent: False, TTL: 01:00:00
Push 5 архивировано.
Кампания 